# Linear Regression for Bitcoin Signal Discovery

A practical guide following the QuantInsti tutorial structure, applied to on-chain metrics.

## Contents
1. What is Linear Regression?
2. Load and Prepare Bitcoin Data
3. Simple Linear Regression with OLS
4. Understanding the Output
5. Checking Assumptions
6. Model Evaluation Metrics
7. Multiple Linear Regression
8. Cycle-by-Cycle Analysis
9. Grid Search for Optimal Threshold

---
## 1. What is Linear Regression?

Linear regression models the relationship between:
- **Y** (dependent variable): What we want to predict → *forward returns*
- **X** (independent variable): What we use to predict → *on-chain metric*

### The Equation

**Simple Linear Regression:**
```
Y = β₀ + β₁X + ε
```

Where:
- `β₀` = intercept (return when X = 0)
- `β₁` = coefficient (how much Y changes per unit X)
- `ε` = error term (what we can't explain)

### For Trading Signals

We're asking: **"If liveliness is X today, what will returns be in 30 days?"**

- **Positive β₁** → Higher metric predicts higher returns → Use `ABOVE` threshold
- **Negative β₁** → Higher metric predicts lower returns → Use `BELOW` threshold

---
## 2. Load and Prepare Bitcoin Data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Statistical libraries
import statsmodels.api as sm
from scipy import stats

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries loaded!")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

# Load our target metric: liveliness
liveliness = pd.read_parquet(DATA_DIR / "liveliness.parquet")
price = pd.read_parquet(DATA_DIR / "price.parquet")

# Rename and merge
liveliness = liveliness.rename(columns={"value": "liveliness"})
price = price.rename(columns={"value": "price"})

df = liveliness.merge(price, on="time").set_index("time").sort_index()

print(f"Data loaded: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
df.head()

In [ ]:
# Create forward returns (Y variable)
# This is what we're trying to predict

FORWARD_DAYS = 30

df['fwd_return'] = df['price'].pct_change(FORWARD_DAYS).shift(-FORWARD_DAYS)

# Check
print(f"Forward return calculated: {FORWARD_DAYS}-day returns")
print(f"\nSample:")
df[['price', 'liveliness', 'fwd_return']].head(10)

In [ ]:
# Quick visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price
ax = axes[0, 0]
ax.plot(df.index, df['price'])
ax.set_title('Bitcoin Price')
ax.set_ylabel('Price (USD)')
ax.set_yscale('log')

# Liveliness
ax = axes[0, 1]
ax.plot(df.index, df['liveliness'], color='orange')
ax.set_title('Liveliness')
ax.set_ylabel('Liveliness')

# Forward returns distribution
ax = axes[1, 0]
df['fwd_return'].dropna().hist(bins=50, ax=ax, alpha=0.7)
ax.axvline(x=0, color='red', linestyle='--')
ax.set_title(f'{FORWARD_DAYS}-Day Forward Return Distribution')
ax.set_xlabel('Return')

# Scatter: liveliness vs returns
ax = axes[1, 1]
ax.scatter(df['liveliness'], df['fwd_return'], alpha=0.3, s=5)
ax.set_title('Liveliness vs Forward Returns')
ax.set_xlabel('Liveliness')
ax.set_ylabel(f'{FORWARD_DAYS}-Day Return')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## 3. Simple Linear Regression with OLS

**OLS = Ordinary Least Squares**

OLS finds the "best-fit" line by minimizing the sum of squared errors:

```
Minimize: Σ(Yᵢ - Ŷᵢ)²
```

This gives us the line that's closest to all the data points.

In [ ]:
# Prepare data for regression
# Drop NaN values
reg_data = df[['liveliness', 'fwd_return']].dropna()

# Define X and Y
X = reg_data['liveliness']
Y = reg_data['fwd_return']

# IMPORTANT: Add constant term for intercept
# Without this, the regression line is forced through the origin
X_with_const = sm.add_constant(X)

print(f"Sample size: {len(reg_data)}")
print(f"\nX (first 5 rows with constant):")
print(X_with_const.head())

In [ ]:
# Fit the OLS model
model = sm.OLS(Y, X_with_const).fit()

# Print full summary
print(model.summary())

---
## 4. Understanding the Output

Let's break down what each part means:

In [ ]:
# Extract key statistics
print("=" * 60)
print("KEY REGRESSION STATISTICS")
print("=" * 60)

# 1. Coefficients
print("\n1. COEFFICIENTS")
print("-" * 40)
print(f"   Intercept (β₀): {model.params['const']:.6f}")
print(f"   Slope (β₁):     {model.params['liveliness']:.6f}")
print(f"\n   Interpretation:")
print(f"   For each 0.01 increase in liveliness,")
print(f"   {FORWARD_DAYS}-day return changes by {model.params['liveliness']*0.01*100:.2f}%")

# Direction
direction = "ABOVE" if model.params['liveliness'] > 0 else "BELOW"
print(f"\n   → Signal direction: Use {direction} threshold")

In [ ]:
# 2. P-value
print("\n2. P-VALUE (Statistical Significance)")
print("-" * 40)
pval = model.pvalues['liveliness']
print(f"   p-value: {pval:.2e}")

if pval < 0.001:
    sig = "*** HIGHLY SIGNIFICANT (p < 0.001)"
elif pval < 0.01:
    sig = "** VERY SIGNIFICANT (p < 0.01)"
elif pval < 0.05:
    sig = "* SIGNIFICANT (p < 0.05)"
elif pval < 0.10:
    sig = ". MARGINALLY SIGNIFICANT (p < 0.10)"
else:
    sig = "NOT SIGNIFICANT (p >= 0.10)"

print(f"   {sig}")
print(f"\n   Interpretation:")
print(f"   There is a {pval*100:.4f}% probability this relationship")
print(f"   occurred by random chance.")

In [ ]:
# 3. R-squared
print("\n3. R-SQUARED (Explained Variance)")
print("-" * 40)
r2 = model.rsquared
print(f"   R²: {r2:.4f} ({r2*100:.2f}%)")
print(f"\n   Interpretation:")
print(f"   Liveliness explains {r2*100:.2f}% of the variance")
print(f"   in {FORWARD_DAYS}-day forward returns.")
print(f"\n   NOTE: In finance, even R² = 1-2% can be profitable!")
print(f"   Markets are noisy. Perfect prediction is impossible.")

In [ ]:
# 4. T-statistic
print("\n4. T-STATISTIC")
print("-" * 40)
tstat = model.tvalues['liveliness']
print(f"   t-stat: {tstat:.2f}")
print(f"\n   Formula: t = coefficient / standard_error")
print(f"            t = {model.params['liveliness']:.6f} / {model.bse['liveliness']:.6f}")
print(f"\n   Rule of thumb: |t| > 2 is significant")
print(f"   Your |t| = {abs(tstat):.2f} → {'SIGNIFICANT' if abs(tstat) > 2 else 'NOT SIGNIFICANT'}")

In [ ]:
# 5. Confidence Intervals
print("\n5. CONFIDENCE INTERVALS")
print("-" * 40)
ci = model.conf_int(alpha=0.05)
print(f"   95% CI for coefficient: [{ci.loc['liveliness', 0]:.6f}, {ci.loc['liveliness', 1]:.6f}]")
print(f"\n   Interpretation:")
print(f"   We're 95% confident the true coefficient is in this range.")

if ci.loc['liveliness', 0] * ci.loc['liveliness', 1] > 0:
    print(f"   The interval does NOT contain zero → Effect is real!")
else:
    print(f"   The interval contains zero → Effect might not be real.")

In [ ]:
# Visualize the regression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter with regression line
ax = axes[0]
ax.scatter(X, Y, alpha=0.3, s=5, label='Data')

# Regression line
x_line = np.linspace(X.min(), X.max(), 100)
y_line = model.params['const'] + model.params['liveliness'] * x_line
ax.plot(x_line, y_line, 'r-', linewidth=2, label='Regression line')

# Confidence interval band
predictions = model.get_prediction(sm.add_constant(x_line))
pred_summary = predictions.summary_frame(alpha=0.05)
ax.fill_between(x_line, pred_summary['mean_ci_lower'], pred_summary['mean_ci_upper'], 
                color='red', alpha=0.2, label='95% CI')

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Liveliness')
ax.set_ylabel(f'{FORWARD_DAYS}-Day Return')
ax.set_title(f'Linear Regression: Liveliness vs Returns\n'
             f'β = {model.params["liveliness"]:.4f}, p = {pval:.2e}, R² = {r2:.4f}')
ax.legend()
ax.set_ylim(-0.5, 1.0)

# Coefficient with CI
ax = axes[1]
ax.barh(['Liveliness'], [model.params['liveliness']], 
        xerr=[[model.params['liveliness'] - ci.loc['liveliness', 0]], 
              [ci.loc['liveliness', 1] - model.params['liveliness']]],
        color='steelblue', capsize=5)
ax.axvline(x=0, color='red', linestyle='--')
ax.set_xlabel('Coefficient Value')
ax.set_title('Coefficient with 95% Confidence Interval\n(Should not cross zero)')

plt.tight_layout()
plt.show()

---
## 5. Checking Assumptions

Linear regression has 5 key assumptions. If violated, results may be misleading.

### The Assumptions:
1. **Linearity** - Relationship is linear
2. **Independence** - Errors are independent (no autocorrelation)
3. **Homoscedasticity** - Constant variance of errors
4. **Normality** - Errors are normally distributed
5. **No Multicollinearity** - Independent variables not highly correlated

In [ ]:
# Calculate residuals
residuals = model.resid
fitted_values = model.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted (check linearity & homoscedasticity)
ax = axes[0, 0]
ax.scatter(fitted_values, residuals, alpha=0.3, s=5)
ax.axhline(y=0, color='red', linestyle='--')
ax.set_xlabel('Fitted Values')
ax.set_ylabel('Residuals')
ax.set_title('1. Residuals vs Fitted\n(Should be random scatter around 0)')

# 2. Q-Q Plot (check normality)
ax = axes[0, 1]
stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title('2. Q-Q Plot\n(Points should follow the line)')

# 3. Histogram of residuals
ax = axes[1, 0]
ax.hist(residuals, bins=50, density=True, alpha=0.7, label='Residuals')
# Overlay normal distribution
x = np.linspace(residuals.min(), residuals.max(), 100)
ax.plot(x, stats.norm.pdf(x, residuals.mean(), residuals.std()), 
        'r-', linewidth=2, label='Normal')
ax.axvline(x=0, color='gray', linestyle='--')
ax.set_xlabel('Residuals')
ax.set_ylabel('Density')
ax.set_title('3. Residual Distribution\n(Should be bell-shaped)')
ax.legend()

# 4. Residuals over time (check autocorrelation)
ax = axes[1, 1]
ax.plot(reg_data.index, residuals, alpha=0.5)
ax.axhline(y=0, color='red', linestyle='--')
ax.set_xlabel('Time')
ax.set_ylabel('Residuals')
ax.set_title('4. Residuals Over Time\n(Should not show patterns)')

plt.tight_layout()
plt.show()

In [ ]:
# Statistical tests for assumptions
print("=" * 60)
print("ASSUMPTION TESTS")
print("=" * 60)

# 1. Normality test (Jarque-Bera)
jb_stat, jb_pval, skew, kurtosis = sm.stats.jarque_bera(residuals)
print(f"\n1. NORMALITY (Jarque-Bera test)")
print(f"   Statistic: {jb_stat:.2f}")
print(f"   p-value: {jb_pval:.2e}")
print(f"   Skewness: {skew:.2f}")
print(f"   Kurtosis: {kurtosis:.2f}")
if jb_pval < 0.05:
    print(f"   ⚠️  Residuals are NOT normally distributed (common in finance)")
else:
    print(f"   ✓ Residuals appear normally distributed")

# 2. Autocorrelation test (Durbin-Watson)
dw_stat = sm.stats.durbin_watson(residuals)
print(f"\n2. AUTOCORRELATION (Durbin-Watson test)")
print(f"   Statistic: {dw_stat:.2f}")
print(f"   (Value near 2 = no autocorrelation)")
if dw_stat < 1.5:
    print(f"   ⚠️  Positive autocorrelation detected")
elif dw_stat > 2.5:
    print(f"   ⚠️  Negative autocorrelation detected")
else:
    print(f"   ✓ No strong autocorrelation")

# 3. Heteroscedasticity test (Breusch-Pagan)
bp_stat, bp_pval, _, _ = sm.stats.diagnostic.het_breuschpagan(residuals, X_with_const)
print(f"\n3. HOMOSCEDASTICITY (Breusch-Pagan test)")
print(f"   Statistic: {bp_stat:.2f}")
print(f"   p-value: {bp_pval:.4f}")
if bp_pval < 0.05:
    print(f"   ⚠️  Heteroscedasticity detected (variance not constant)")
else:
    print(f"   ✓ Homoscedasticity holds")

---
## 6. Model Evaluation Metrics

In [ ]:
# Calculate additional metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_pred = model.fittedvalues
y_true = Y

print("=" * 60)
print("MODEL EVALUATION METRICS")
print("=" * 60)

print(f"\nR-squared:           {model.rsquared:.4f}")
print(f"Adjusted R-squared:  {model.rsquared_adj:.4f}")
print(f"F-statistic:         {model.fvalue:.2f}")
print(f"F-statistic p-value: {model.f_pvalue:.2e}")

mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, y_pred)

print(f"\nMean Squared Error:  {mse:.6f}")
print(f"Root MSE:            {rmse:.4f} ({rmse*100:.2f}%)")
print(f"Mean Absolute Error: {mae:.4f} ({mae*100:.2f}%)")

# AIC and BIC (lower is better)
print(f"\nAIC: {model.aic:.2f}")
print(f"BIC: {model.bic:.2f}")

---
## 7. Multiple Linear Regression

Let's add more metrics to see if we can improve the model.

In [ ]:
# Load additional metrics
metrics_to_load = ['mvrv', 'nupl', 'sopr', 'nvt', 'supply_lth_sth_ratio']

for metric in metrics_to_load:
    try:
        temp = pd.read_parquet(DATA_DIR / f"{metric}.parquet")
        temp = temp.rename(columns={"value": metric}).set_index("time")
        df = df.join(temp, how='left')
        print(f"Loaded: {metric}")
    except:
        print(f"Could not load: {metric}")

print(f"\nColumns available: {list(df.columns)}")

In [ ]:
# Check correlation between predictors (multicollinearity)
predictors = ['liveliness', 'mvrv', 'nupl', 'sopr', 'nvt', 'supply_lth_sth_ratio']
available_predictors = [p for p in predictors if p in df.columns]

corr_matrix = df[available_predictors].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
            vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation Matrix (check for multicollinearity)\nHigh correlation (>0.7) between predictors is problematic')
plt.tight_layout()
plt.show()

In [ ]:
# Multiple regression with selected predictors
# Choose predictors that are not highly correlated
selected_predictors = ['liveliness', 'sopr', 'nvt']
selected_predictors = [p for p in selected_predictors if p in df.columns]

# Prepare data
multi_data = df[selected_predictors + ['fwd_return']].dropna()

X_multi = multi_data[selected_predictors]
Y_multi = multi_data['fwd_return']
X_multi_const = sm.add_constant(X_multi)

# Fit multiple regression
model_multi = sm.OLS(Y_multi, X_multi_const).fit()

print(model_multi.summary())

In [ ]:
# Compare models
print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(f"\n{'Metric':<20} {'Simple':>15} {'Multiple':>15}")
print("-" * 50)
print(f"{'R-squared':<20} {model.rsquared:>15.4f} {model_multi.rsquared:>15.4f}")
print(f"{'Adjusted R²':<20} {model.rsquared_adj:>15.4f} {model_multi.rsquared_adj:>15.4f}")
print(f"{'AIC':<20} {model.aic:>15.2f} {model_multi.aic:>15.2f}")
print(f"{'BIC':<20} {model.bic:>15.2f} {model_multi.bic:>15.2f}")

print(f"\nLower AIC/BIC is better.")
if model_multi.aic < model.aic:
    print(f"→ Multiple regression is preferred by AIC.")
else:
    print(f"→ Simple regression is preferred by AIC (simpler is better).")

---
## 8. Cycle-by-Cycle Analysis

**Critical for trading:** A signal must work across multiple market cycles, not just historically.

In [ ]:
# Define bull market periods
bull_periods = [
    {"name": "2015-2017", "start": "2015-10-01", "end": "2017-12-17"},
    {"name": "2019", "start": "2018-12-15", "end": "2019-06-26"},
    {"name": "2020-2021", "start": "2020-03-13", "end": "2021-11-10"},
    {"name": "2023-2024", "start": "2022-11-21", "end": "2024-03-14"},
    {"name": "2024-Present", "start": "2024-09-01", "end": "2026-12-31"},
]

# Run regression for each cycle
cycle_results = []

for period in bull_periods:
    # Filter data
    mask = (df.index >= period['start']) & (df.index < period['end'])
    cycle_data = df.loc[mask, ['liveliness', 'fwd_return']].dropna()
    
    if len(cycle_data) < 30:
        continue
    
    # Run regression
    X_cycle = sm.add_constant(cycle_data['liveliness'])
    Y_cycle = cycle_data['fwd_return']
    
    model_cycle = sm.OLS(Y_cycle, X_cycle).fit()
    
    cycle_results.append({
        'cycle': period['name'],
        'n': len(cycle_data),
        'coefficient': model_cycle.params['liveliness'],
        'p_value': model_cycle.pvalues['liveliness'],
        'r_squared': model_cycle.rsquared,
        'significant': model_cycle.pvalues['liveliness'] < 0.1
    })

cycle_df = pd.DataFrame(cycle_results)
print(cycle_df.to_string(index=False))

In [ ]:
# Visualize cycle results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coefficient by cycle
ax = axes[0]
colors = ['green' if c > 0 else 'red' for c in cycle_df['coefficient']]
ax.bar(cycle_df['cycle'], cycle_df['coefficient'], color=colors, alpha=0.7)
ax.axhline(y=0, color='black', linestyle='--')
ax.set_xlabel('Bull Market Cycle')
ax.set_ylabel('Coefficient')
ax.set_title('Liveliness Coefficient by Cycle\n(Should be same sign across cycles)')
ax.tick_params(axis='x', rotation=45)

# Significance
ax = axes[1]
colors = ['green' if p < 0.1 else 'gray' for p in cycle_df['p_value']]
ax.bar(cycle_df['cycle'], -np.log10(cycle_df['p_value']), color=colors, alpha=0.7)
ax.axhline(y=-np.log10(0.1), color='red', linestyle='--', label='p=0.1')
ax.axhline(y=-np.log10(0.05), color='orange', linestyle='--', label='p=0.05')
ax.set_xlabel('Bull Market Cycle')
ax.set_ylabel('-log10(p-value)')
ax.set_title('Statistical Significance by Cycle\n(Higher = more significant)')
ax.tick_params(axis='x', rotation=45)
ax.legend()

plt.tight_layout()
plt.show()

# Summary
n_positive = (cycle_df['coefficient'] > 0).sum()
n_negative = (cycle_df['coefficient'] < 0).sum()
n_significant = cycle_df['significant'].sum()

print(f"\n" + "=" * 60)
print("CYCLE CONSISTENCY SUMMARY")
print("=" * 60)
print(f"Positive coefficients: {n_positive}/{len(cycle_df)}")
print(f"Negative coefficients: {n_negative}/{len(cycle_df)}")
print(f"Significant (p<0.1):   {n_significant}/{len(cycle_df)}")
print(f"\nSign consistency: {max(n_positive, n_negative)/len(cycle_df)*100:.0f}%")

---
## 9. Grid Search for Optimal Threshold

Now we know the direction (coefficient sign). Let's find the optimal threshold.

In [ ]:
# Determine direction from coefficient
avg_coef = cycle_df['coefficient'].mean()
direction = "below" if avg_coef < 0 else "above"

print(f"Average coefficient across cycles: {avg_coef:.4f}")
print(f"Direction: Buy when liveliness is {direction.upper()} threshold")

In [ ]:
# Grid search
# Test thresholds from 5th to 95th percentile

# Use post-2018 data only (more recent cycles)
recent_data = df[df.index >= '2018-12-15'][['liveliness', 'fwd_return']].dropna()

percentiles = np.linspace(5, 95, 19)
thresholds = [recent_data['liveliness'].quantile(p/100) for p in percentiles]

grid_results = []

for thresh in thresholds:
    # Apply signal
    if direction == "below":
        signal_returns = recent_data[recent_data['liveliness'] < thresh]['fwd_return']
    else:
        signal_returns = recent_data[recent_data['liveliness'] > thresh]['fwd_return']
    
    if len(signal_returns) < 20:
        continue
    
    # Calculate metrics
    mean_ret = signal_returns.mean()
    std_ret = signal_returns.std()
    sharpe = mean_ret / std_ret * np.sqrt(12) if std_ret > 0 else 0
    win_rate = (signal_returns > 0).mean()
    
    grid_results.append({
        'threshold': thresh,
        'n_trades': len(signal_returns),
        'mean_return': mean_ret,
        'sharpe': sharpe,
        'win_rate': win_rate
    })

grid_df = pd.DataFrame(grid_results)
grid_df

In [ ]:
# Plot Sharpe curve - should be SMOOTH
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sharpe curve
ax = axes[0, 0]
ax.plot(grid_df['threshold'], grid_df['sharpe'], 'b-o', markersize=6)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
best_idx = grid_df['sharpe'].idxmax()
ax.axvline(x=grid_df.loc[best_idx, 'threshold'], color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Threshold')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('SHARPE CURVE (should be smooth!)\nSpiky = overfit, Smooth = robust')

# Mean return curve
ax = axes[0, 1]
ax.plot(grid_df['threshold'], grid_df['mean_return']*100, 'g-o', markersize=6)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Threshold')
ax.set_ylabel('Mean Return (%)')
ax.set_title('Mean Return by Threshold')

# Win rate curve
ax = axes[1, 0]
ax.plot(grid_df['threshold'], grid_df['win_rate']*100, 'orange', marker='o', markersize=6)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Threshold')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Win Rate by Threshold')

# Number of trades
ax = axes[1, 1]
ax.bar(grid_df['threshold'], grid_df['n_trades'], width=0.01, alpha=0.7)
ax.set_xlabel('Threshold')
ax.set_ylabel('Number of Trades')
ax.set_title('Sample Size by Threshold')

plt.tight_layout()
plt.show()

# Smoothness score
sharpe_diffs = np.diff(grid_df['sharpe'])
smoothness = np.std(sharpe_diffs) / np.mean(np.abs(grid_df['sharpe']))
print(f"\nSmoothness score: {smoothness:.2f}")
print(f"(Lower is better. < 0.5 is acceptably smooth)")
print(f"→ {'✓ SMOOTH' if smoothness < 0.5 else '✗ SPIKY (possible overfit)'}")

In [ ]:
# Best threshold
best = grid_df.loc[grid_df['sharpe'].idxmax()]

print("=" * 60)
print("OPTIMAL SIGNAL")
print("=" * 60)
print(f"\nMetric: liveliness")
print(f"Direction: {direction}")
print(f"Threshold: {best['threshold']:.4f}")
print(f"\nExpected Performance:")
print(f"  Sharpe Ratio: {best['sharpe']:.2f}")
print(f"  Mean Return:  {best['mean_return']*100:.2f}%")
print(f"  Win Rate:     {best['win_rate']*100:.1f}%")
print(f"  Sample Size:  {best['n_trades']:.0f} periods")
print(f"\nSignal: Buy when liveliness {direction} {best['threshold']:.4f}")

---
## Summary

### What We Learned

1. **Linear Regression** models the relationship between metrics and future returns
2. **Key Statistics**: coefficient (direction), p-value (significance), R² (fit)
3. **Assumptions** must be checked: linearity, normality, homoscedasticity
4. **Cycle Analysis** is critical: signal must work across multiple periods
5. **Sharpe Curve** should be smooth to avoid overfitting

### The Workflow

```
1. Run regression → Get coefficient sign (direction)
2. Check p-value → Is it significant?
3. Check assumptions → Are results valid?
4. Test by cycle → Does it work consistently?
5. Grid search → Find optimal threshold
6. Check smoothness → Is it robust or overfit?
```

### Next Steps

Run `python -m src.walk_forward` to validate this signal out-of-sample!